# Real Market Data Backtesting with q/KDB-X

This notebook demonstrates how to build a **financial backtesting system** using KDB-X/q. We'll simulate a simple trading strategy and calculate how much profit or loss it would have generated.

## What we explore in this Notebook?


- **Real market data** from Yahoo Finance (last 8 days, minute-level)
- **Moving average crossover strategy** - a classic momentum approach
- **15-minute bars** - aggregated from minute data for cleaner signals
- **Complete trade analysis** - PnL, win rate, drawdowns, and more

## The Strategy: Moving Average Crossover

We'll implement a **dual moving average crossover**:
- **Fast MA**: 10-period moving average (responsive to recent price action)
- **Slow MA**: 30-period moving average (tracks longer-term trend)
- **Buy signal**: When fast MA crosses *above* slow MA → bullish momentum
- **Sell signal**: When fast MA crosses *below* slow MA → bearish momentum

This is a trend-following strategy used by traders to ride momentum waves.

## Prerequisites

- Requires KDB-X to be installed, you can sign up and download on [Developer Center](https://developer.kx.com/products/kdb-x/install). For full install instructions see: [KDB-X Install](https://code.kx.com/kdb-x/).

- To Install [KDB-X Python](https://code.kx.com/pykx/examples/jupyter-integration.html): `pip install --upgrade pykx` (This is required to run q in a Python notebook, but you can run the q directly from a q session if you prefer.)



In [3]:
!pip install -qq --upgrade pykx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 72.2 MB/s eta 0:00:00


In [ ]:
import pykx as kx
import yfinance as yf
from datetime import datetime, timedelta

## Step 1: Fetch Real Market Data

We'll download the last 8 days of minute-level data for **AAPL** and **MSFT**. Yahoo Finance provides free intraday data, though it's limited to the most recent period.

### Why minute data?
Minute bars give us fine-grained price action, allowing us to:
- Aggregate into any timeframe we want (5min, 15min, 1hr)
- Simulate realistic entry/exit timing
- See intraday volatility patterns

We'll use Python/yfinance to fetch the data, then convert it to a q table for all subsequent analysis.

In [3]:
# Calculate dynamic date range (last 8 days)
end_date = datetime.now()
start_date = end_date - timedelta(days=8)

print(f"Fetching data from {start_date.date()} to {end_date.date()}...")

# Download minute-level data
data = yf.download(
    tickers=["AAPL", "MSFT"],
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    interval="1m",
    auto_adjust=True
)

# Reshape from wide (columns per ticker) to long format (rows per ticker)
data_long = data.stack(level=1).reset_index()

# Rename columns to match q conventions
data_long = data_long.rename(columns={
    'Datetime': 'dt',
    'Ticker': 'sym'
})

# Convert ticker symbols to lowercase (q convention)
data_long['sym'] = data_long['sym'].str.lower()

# Convert to q table - this is our bridge from Python to q
quotes = kx.toq(data_long)
kx.q['quotes'] = quotes

print(f"✓ Loaded {len(data_long):,} minute bars")
print(f"✓ Tickers: {', '.join(data_long['sym'].unique())}")

Fetching data from 2026-10-11 to 2026-10-19...
✓ Loaded 3,900 minute bars
✓ Tickers: aapl, msft


## Step 2: Inspect the Data Structure

Let's examine what we're working with. From here on, we use **pure q code** - no more Python!

The data should have columns: `dt` (datetime), `sym` (symbol), `Open`, `High`, `Low`, `Close`, `Volume`.

In [4]:
# Enable q-first mode - write pure q code in subsequent cells
kx.util.jupyter_qfirst_enable()

KDB-X Python now running in 'jupyter_qfirst' mode. All cells by default will be run as q code.
Include '%%py' at the beginning of each cell to run as python code. 


In [5]:
/ Adjust console width for visibility
\c 20 200

In [6]:
/ Show first 10 rows
10 sublist quotes

dt                            sym  Close    High     Low      Open     Volume 
------------------------------------------------------------------------------
2026.10.12D13:30:00.000000000 aapl 340.8    341.47   337.6    338.16   1245538
2026.10.12D13:30:00.000000000 msft 539.2    539.54   537.03   538.07   724504 
2026.10.12D13:31:00.000000000 aapl 341      341.465  339.76   340.845  294777 
2026.10.12D13:31:00.000000000 msft 537.055  539.45   536.5    539.06   95894  
2026.10.12D13:32:00.000000000 aapl 342.1856 342.32   340.71   341.03   361426 
2026.10.12D13:32:00.000000000 msft 536.51   537.52   535.7013 537.1799 58247  
2026.10.12D13:33:00.000000000 aapl 341.65   342.3    341.4    342.22   241556 
2026.10.12D13:33:00.000000000 msft 538.8725 539.0999 536.49   536.49   73845  
2026.10.12D13:34:00.000000000 aapl 341.74   341.83   341.1    341.65   224156 
2026.10.12D13:34:00.000000000 msft 538.89   539.01   538.1901 538.97   64735  


In [7]:
/ Get data summary by symbol
select
  bars:count i,
  start:min dt,
  end:max dt,
  low_price:min Close,
  high_price:max Close
  by sym from quotes



sym | bars start                         end                           low_price high_price
----| -------------------------------------------------------------------------------------
aapl| 1950 2026.10.12D13:30:00.000000000 2026.10.16D19:59:00.000000000 339.135   347.836   
msft| 1950 2026.10.12D13:30:00.000000000 2026.10.16D19:59:00.000000000 524.52    550.985   


## Step 3: Aggregate to 15-Minute Bars

Minute data can be noisy. We'll aggregate it into **15-minute OHLCV bars** for cleaner signals.

### The Process:
1. Extract the time component from each datetime
2. Use [`xbar`](https://code.kx.com/q/ref/xbar/) to bucket times into 15-minute intervals
3. Aggregate: first Open, max High, min Low, last Close, sum Volume
4. Group by symbol and time bucket

This is standard OHLC bar construction used throughout finance.

In [ ]:
/ Extract date and time components
quotes:update date:`date$dt, time:`time$dt from quotes;

/ Create 15-minute time buckets using xbar
/ xbar rounds down to the nearest interval (e.g., 14:37 -> 14:30)
quotes:update bucket:15 xbar `minute$time from quotes;

/ Aggregate to 15-minute bars
bars:select
  open:first Open,
  high:max High,
  low:min Low,
  close:last Close,
  volume:sum Volume
  by sym, date, bucket from quotes;

/ Recreate datetime column and clean up
bars:update dt:date+bucket from bars;
bars:delete date,bucket from bars;

/ Sort by symbol and datetime for efficiency
bars:`sym`dt xasc bars;

"15-minute bars created:"
10 sublist bars

15-minute bars created:
sym  open     high     low      close   volume  dt                           
-----------------------------------------------------------------------------
aapl 338.16   342.32   337.6    341.27  3498896 2026.10.12D13:30:00.000000000
aapl 341.27   342.1    340.33   341.05  1286949 2026.10.12D13:45:00.000000000
aapl 341.08   342.63   341.016  341.885 1446397 2026.10.12D14:00:00.000000000
aapl 341.875  342.11   341.19   341.36  719649  2026.10.12D14:15:00.000000000
aapl 341.36   341.579  340.6075 340.835 668248  2026.10.12D14:30:00.000000000
aapl 340.8    341.3888 340.78   340.795 547310  2026.10.12D14:45:00.000000000
aapl 340.81   341.21   340.5    340.73  525967  2026.10.12D15:00:00.000000000
aapl 340.7301 340.84   339.84   340.065 786073  2026.10.12D15:15:00.000000000
aapl 340.05   340.055  339.18   339.715 746818  2026.10.12D15:30:00.000000000
aapl 339.7225 340.22   339.61   339.83  521991  2026.10.12D15:45:00.000000000


## Step 4: Calculate Moving Averages

Now we calculate our strategy indicators:
- **Fast MA**: 10-period moving average using [`mavg`](https://code.kx.com/q/ref/avg/#mavg)
- **Slow MA**: 30-period moving average

The `by sym` clause ensures we calculate MAs separately for each ticker - no data leakage between symbols!

A 10-period moving average is the average closing price of the last 10 fifteen-minute bars (fast-moving, reacts quickly to recent price changes), while a 30-period moving average is the average closing price of the last 30 fifteen-minute bars (slow-moving, smooths out short-term fluctuations to show the longer-term trend).

### Identifying Crossovers
A crossover occurs when:
- **Bullish**: Fast MA was ≤ Slow MA, now Fast MA > Slow MA
- **Bearish**: Fast MA was ≥ Slow MA, now Fast MA < Slow MA

We use [`prev`](https://code.kx.com/q/ref/next/) to access the previous row's values for comparison.

In [9]:
/ Calculate moving averages per symbol
bars:update
  ma_fast:mavg[10;close],
  ma_slow:mavg[30;close]
  by sym from bars;

/ Get previous values for crossover detection
bars:update
  prev_fast:prev ma_fast,
  prev_slow:prev ma_slow
  by sym from bars;

/ Identify crossover signals
/ signal = 1: bullish cross (BUY)
/ signal = -1: bearish cross (SELL)
/ signal = 0: no signal
bars:update signal:0 from bars;
bars:update signal:1 from bars where (ma_fast>ma_slow) and prev_fast<=prev_slow;
bars:update signal:-1 from bars where (ma_fast<ma_slow) and prev_fast>=prev_slow;

"Trading signals generated:"
select from bars where signal<>0

Trading signals generated:
sym  open     high     low      close    volume  dt                            ma_fast  ma_slow  prev_fast prev_slow signal
---------------------------------------------------------------------------------------------------------------------------
aapl 339.81   340.44   339.8    340.35   447671  2026.10.12D16:00:00.000000000 340.6615 340.7168 340.7535  340.7535  -1    
aapl 342.135  345.29   341.12   345.23   2212024 2026.10.13D13:45:00.000000000 340.8143 340.4803 340.2313  340.3044  1     
aapl 342.2    342.6    341.7    342.4756 647871  2026.10.13D19:00:00.000000000 342.8701 343.1758 343.094   343.0773  -1    
aapl 345.24   346.07   344.76   345.855  1162850 2026.10.14D15:00:00.000000000 344.284  344.1297 343.9461  344.1212  1     
aapl 341.93   342.91   341.72   342.855  855752  2026.10.14D18:00:00.000000000 343.5695 343.737  343.88    343.7595  -1    
aapl 344.67   345.3    344.08   344.78   1104520 2026.10.15D14:45:00.000000000 344.3655 344.2487 344.1945

## Step 5: Generate Trade Pairs

We have signals, now we need to pair them up into actual trades:
- Each **BUY signal** (signal=1) is an entry
- Each **SELL signal** (signal=-1) is an exit
- We match each entry with the next exit using an asof join

### Why Asof Join?
The [`aj`](https://code.kx.com/q/ref/aj/) function finds the "most recent match as of a time" - perfect for matching "next exit after this entry".

For speed, `aj` needs the table it searches (the buys) sorted by time within sym, with the [grouped attribute](https://code.kx.com/q/ref/aj/#performance) (`` `g# ``) applied to `sym`.

We'll trade 100 shares per position for simplicity.

In [10]:
/ Extract buy and sell signals
trades:select sym, dt, signal, close from bars where signal<>0;
trades:update tradetype:`buy from trades where signal=1;
trades:update tradetype:`sell from trades where signal=-1;

/ Separate buys and sells
buys:select sym, entry_dt:dt, entry_price:close from trades where tradetype=`buy;
sells:select sym, exit_dt:dt, exit_price:close from trades where tradetype=`sell;

/ aj matches on the same column names in both tables, so give buys an exit_dt column holding its entry time
buys:update exit_dt:entry_dt from buys;

/ aj is fastest when the table being searched (buys) is sorted by sym then time
/ and has the grouped attribute on sym
buys:update `g#sym from `sym`exit_dt xasc buys;
sells:`sym`exit_dt xasc sells;

/ Match each SELL with its most recent preceding BUY using asof join (backward-looking)
positions:aj[`sym`exit_dt;sells;buys];
positions:select sym, entry_dt, entry_price, exit_dt, exit_price from positions;

/ Remove incomplete trades where no matching buy was found (null entry_price)
positions:delete from positions where null entry_price;

/ Set position size
positions:update size:100 from positions;

"Generated positions:"
positions

Generated positions:
sym  entry_dt                      entry_price exit_dt                       exit_price size
--------------------------------------------------------------------------------------------
aapl 2026.10.13D13:45:00.000000000 345.23      2026.10.13D19:00:00.000000000 342.4756   100 
aapl 2026.10.14D15:00:00.000000000 345.855     2026.10.14D18:00:00.000000000 342.855    100 
aapl 2026.10.15D14:45:00.000000000 344.78      2026.10.15D18:30:00.000000000 342.335    100 
aapl 2026.10.16D15:00:00.000000000 345.9095    2026.10.16D18:15:00.000000000 341.795    100 
msft 2026.10.12D16:00:00.000000000 541.715     2026.10.12D18:15:00.000000000 539.2902   100 
msft 2026.10.13D13:30:00.000000000 549.41      2026.10.14D13:30:00.000000000 543.18     100 
msft 2026.10.15D15:30:00.000000000 537.205     2026.10.16D14:45:00.000000000 534.48     100 


## Step 6: Calculate PnL

For each position, we calculate:
- **PnL in dollars**: `size × (exit_price - entry_price)`
- **PnL percentage**: `100 × (exit_price - entry_price) / entry_price`
- **Holding period**: How long we held the position

This tells us which trades were winners and losers.

In [11]:
/ Calculate PnL metrics for each trade
positions:update
  pnl:size * exit_price - entry_price,
  pnl_pct:100 * (exit_price - entry_price) % entry_price,
  holding_period:exit_dt - entry_dt
  from positions;

"Position PnL: "
positions

Position PnL: 
sym  entry_dt                      entry_price exit_dt                       exit_price size pnl       pnl_pct    holding_period      
--------------------------------------------------------------------------------------------------------------------------------------
aapl 2026.10.13D13:45:00.000000000 345.23      2026.10.13D19:00:00.000000000 342.4756   100  -275.4425 -0.7978521 0D05:15:00.000000000
aapl 2026.10.14D15:00:00.000000000 345.855     2026.10.14D18:00:00.000000000 342.855    100  -300      -0.8674155 0D03:00:00.000000000
aapl 2026.10.15D14:45:00.000000000 344.78      2026.10.15D18:30:00.000000000 342.335    100  -244.5007 -0.70915   0D03:45:00.000000000
aapl 2026.10.16D15:00:00.000000000 345.9095    2026.10.16D18:15:00.000000000 341.795    100  -411.4471 -1.189465  0D03:15:00.000000000
msft 2026.10.12D16:00:00.000000000 541.715     2026.10.12D18:15:00.000000000 539.2902   100  -242.4805 -0.4476163 0D02:15:00.000000000
msft 2026.10.13D13:30:00.000000000 549.4

In [12]:
/ Extract the pnl values
pnl:positions`pnl;

## Step 7: Performance by Symbol

Different stocks behave differently. Let's break down performance by ticker to see which one the strategy works better on.

We might find:
- One stock trends more consistently (better for MA crossover)
- One stock is choppier (more whipsaws, lower win rate)
- Differences in volatility and trade frequency

In [13]:
/ Aggregate statistics by symbol
symbol_stats:select
  trades:count i,
  winners:sum pnl>0,
  total_pnl:sum pnl,
  avg_pnl:avg pnl,
  win_rate:100 * (sum pnl>0) % count i,
  best_trade:max pnl,
  worst_trade:min pnl
  by sym from positions;

-1 "\n=== PERFORMANCE BY SYMBOL ===";
symbol_stats

sym | trades winners total_pnl avg_pnl   win_rate best_trade worst_trade
----| ------------------------------------------------------------------
aapl| 4      0       -1231.39  -307.8476 0        -244.5007  -411.4471  
msft| 3      0       -1137.982 -379.3274 0        -242.4805  -622.998   

=== PERFORMANCE BY SYMBOL ===


## Step 8: Cumulative PnL Over Time

To understand the strategy's progression, we calculate cumulative PnL - how our account balance would have grown (or shrunk) over the backtest period.

The [`sums`](https://code.kx.com/q/ref/sum/#sums) function creates a running total, perfect for equity curves.

In [14]:
/ Sort trades chronologically and calculate running PnL
pnl_timeline:`entry_dt xasc select entry_dt, sym, pnl from positions;
pnl_timeline:update cumulative_pnl:sums pnl from pnl_timeline;

"CUMULATIVE PnL OVER TIME: "
-15 sublist pnl_timeline

CUMULATIVE PnL OVER TIME: 
entry_dt                      sym  pnl       cumulative_pnl
-----------------------------------------------------------
2026.10.12D16:00:00.000000000 msft -242.4805 -242.4805     
2026.10.13D13:30:00.000000000 msft -622.998  -865.4785     
2026.10.13D13:45:00.000000000 aapl -275.4425 -1140.921     
2026.10.14D15:00:00.000000000 aapl -300      -1440.921     
2026.10.15D14:45:00.000000000 aapl -244.5007 -1685.422     
2026.10.15D15:30:00.000000000 msft -272.5037 -1957.925     
2026.10.16D15:00:00.000000000 aapl -411.4471 -2369.373     


## Step 9: Maximum Drawdown Analysis

**Maximum drawdown** is the largest peak-to-trough decline in cumulative PnL. It measures the strategy's worst-case risk.

### Why it matters:
- Shows how much capital you could have lost at the worst point
- Helps determine position sizing and risk tolerance
- More realistic than just looking at final PnL

We calculate:
1. Running maximum PnL at each point (the peak)
2. Current drawdown = Current PnL - Running maximum
3. Maximum drawdown = Largest negative value

In [15]:
/ Calculate running maximum (the peak)
pnl_timeline:update running_max:maxs cumulative_pnl from pnl_timeline

/ Calculate drawdown from peak
pnl_timeline:update drawdown:cumulative_pnl - running_max from pnl_timeline

/ Find maximum drawdown
max_dd:min pnl_timeline[`drawdown]
"Maximum Drawdown: $", string max_dd

/ Show when it occurred
"Worst drawdown period:"
select from pnl_timeline where drawdown = min drawdown

Maximum Drawdown: $-2126.892
Worst drawdown period:
entry_dt                      sym  pnl       cumulative_pnl running_max drawdown 
---------------------------------------------------------------------------------
2026.10.16D15:00:00.000000000 aapl -411.4471 -2369.373      -242.4805   -2126.892


## Step 10: Testing Different Parameters

What if we used **different MA periods**? Let's try a faster 5/20 combination and compare signal frequency.

### Trade-offs:
- **Shorter MAs** (5/20): More signals, more responsive, but more whipsaws
- **Longer MAs** (10/30): Fewer signals, smoother, but slower to react

This is called **parameter sensitivity analysis** - testing how results change with different settings.

In [16]:
/ Calculate alternative moving averages (5/20)
bars2:update
  ma_fast5:mavg[5;close],
  ma_slow20:mavg[20;close]
  by sym from bars;

/ Detect crossovers with new parameters
/ To detect a "crossover," we need to know the values from the PREVIOUS row.
/ 'prev' shifts the data down by one, allowing us to compare 'now' vs 'then'.
bars2:update
  prev_fast5:prev ma_fast5,
  prev_slow20:prev ma_slow20
  by sym from bars2;

/ Initialize all signals to 0 (No Trade/Hold)
bars2:update signal2:0 from bars2;

/ BULLISH CROSSOVER:
/ Fast MA is currently ABOVE slow MA, but was BELOW or EQUAL in the previous bar.
bars2:update signal2:1 from bars2 where (ma_fast5>ma_slow20) and prev_fast5<=prev_slow20;

/ BEARISH CROSSOVER:
/ Fast MA is currently BELOW slow MA, but was ABOVE or EQUAL in the previous bar
bars2:update signal2:-1 from bars2 where (ma_fast5<ma_slow20) and prev_fast5>=prev_slow20;

/ Compare signal counts
signal_count:([]
  strategy:`$("MA_10_30";"MA_5_20");
  signals:(sum abs bars[`signal]; sum abs bars2[`signal2])
  );

"SIGNAL FREQUENCY COMPARISON: "
signal_count

SIGNAL FREQUENCY COMPARISON: 
strategy signals
----------------
MA_10_30 15     
MA_5_20  18     


#### Challenge: Can you find the PnL stats for this new 5/20 Moving Average strategy?


In [17]:
/ Challenge Code:

## Conclusion and Next Steps

### What We Built

This notebook demonstrated:
1. **Real data integration**: Yahoo Finance → q/KDB-X pipeline
2. **Bar aggregation**: Minute data → 15-minute OHLC bars
3. **Technical indicators**: Moving averages with `mavg`
4. **Signal generation**: Crossover detection logic
5. **Trade matching**: Entry/exit pairing with `aj`
6. **Performance analysis**: PnL, win rate, drawdown, cumulative returns
7. **Parameter testing**: Comparing different MA periods

### Key q/KDB-X Features Used

- **`xbar`**: Time bucketing for bar aggregation
- **`mavg`**: Efficient moving averages
- **`by` clause**: Per-symbol calculations without loops
- **`aj` (asof join)**: Time-series matching for trade pairing
- **`sums` / `maxs`**: Running calculations for equity curves
- **`prev`**: Access previous row values for crossover detection
- **Attributes (`` `g# ``)**: Grouped attribute on `sym` to speed up `aj` lookups

### Improvements for Production

To make this a real trading strategy, add:

1. **Transaction costs**: Commissions ($0.005/share?), slippage (1-2 bps?)
2. **Position sizing**: Risk-based sizing, not fixed 100 shares
3. **Risk management**: Stop losses, maximum position limits
4. **Multiple timeframes**: Confirm signals on higher timeframes
5. **Volume filters**: Ignore signals on low-volume bars
6. **Market regime detection**: Different parameters for trending vs ranging markets
7. **Walk-forward testing**: Test on rolling windows to avoid overfitting
8. **Realistic execution**: Model market impact, time-in-force

### Why q/KDB-X?

This example used just 8 days of minute data. In production:
- Backtests run on **years of tick data** (billions of records)
- Real-time strategies process **millions of quotes per second**
- Portfolio analytics aggregate **thousands of instruments simultaneously**

q/KDB-X excels at these scales with the same clean syntax we used here. The code patterns are identical whether you're processing 10,000 rows or 10 billion.

### Resources

- [q Reference Card](https://code.kx.com/q/ref/)
- [Time-series joins (aj, asof)](https://code.kx.com/q/ref/aj/)
- [Moving averages (mavg)](https://code.kx.com/q/ref/avg/#mavg)
- [KDB-X Python Documentation](https://code.kx.com/pykx/)

Happy backtesting! 🚀